# New Notebook file

In [ ]:
import random
import string

from pyspark.sql import SparkSession
from pyspark.sql.functions import abs, avg, array, explode, col

spark = (
    SparkSession.builder
    .appName("Skewed Joins")
    .master("local[*]")
    .config("spark.sql.autoBroadcastJoinThreshold", -1)
    .getOrCreate()
)

In [ ]:
# Data generator - mirrors DataGenerator.scala
# 50% of data is Razer Blade (index 0) to simulate skew

LAPTOP_MODELS = [
    ("Razer", "Blade"),
    ("Alienware", "Area-51"),
    ("HP", "Omen"),
    ("Acer", "Predator"),
    ("Asus", "ROG"),
    ("Lenovo", "Legion"),
    ("MSI", "Raider"),
]

def random_laptop_model(uniform=False):
    if not uniform and random.random() < 0.5:
        return LAPTOP_MODELS[0]
    return random.choice(LAPTOP_MODELS)

def random_proc_speed():
    return float(f"3.{random.randint(0, 8)}")

def random_registration():
    return "".join(random.choices(string.ascii_letters + string.digits, k=7))

def random_price():
    return 500 + random.randint(0, 1499)

def random_laptop():
    make, model = random_laptop_model()
    return (random_registration(), make, model, random_proc_speed())

def random_laptop_offer():
    make, model = random_laptop_model()
    return (make, model, random_proc_speed(), float(random_price()))



## Problem
An online store sells gaming laptops. Two laptops are "similar" if they share the same make & model and their CPU speeds are within 0.1 GHz of each other.

**Goal:** For each laptop (by registration), find the average sale price of all "similar" offers.

The data is skewed â€” ~50% of records are Razer Blade.

In [ ]:
# Generate datasets
laptops = spark.createDataFrame(
    [random_laptop() for _ in range(40000)],
    schema=["registration", "make", "model", "procSpeed"]
)

laptop_offers = spark.createDataFrame(
    [random_laptop_offer() for _ in range(100000)],
    schema=["make", "model", "procSpeed", "salePrice"]
)

laptops.printSchema()
laptop_offers.printSchema()

## Approach 1: Naive join with post-filter
Join on `make` + `model`, then filter by `abs(procSpeed diff) <= 0.1`.

This forces a SortMergeJoin on just two columns, landing most data in the same partitions (skew), then does a wide filter pass.

In [ ]:
joined = (
    laptops.join(laptop_offers, on=["make", "model"])
    .filter(abs(laptop_offers["procSpeed"] - laptops["procSpeed"]) <= 0.1)
    .groupBy("registration")
    .agg(avg("salePrice").alias("averagePrice"))
)

joined.explain()
joined.show()

## Approach 2: Explode + equi-join (optimized)
Explode each laptop's `procSpeed` into three values: `speed - 0.1`, `speed`, `speed + 0.1`.

Now we can join on `make`, `model`, AND `procSpeed` â€” a true equi-join with three keys. The join condition becomes exact, distributing load more evenly across partitions.

This trades a 3x row expansion on the smaller table for a much cheaper equi-join instead of a filtered non-equi join.

In [ ]:
laptops2 = laptops.withColumn(
    "procSpeed",
    explode(array(
        col("procSpeed") - 0.1,
        col("procSpeed"),
        col("procSpeed") + 0.1,
    ))
)

joined2 = (
    laptops2.join(laptop_offers, on=["make", "model", "procSpeed"])
    .groupBy("registration")
    .agg(avg("salePrice").alias("averagePrice"))
)

joined2.explain()
joined2.show()